# logfire_api smoke test

`logfire_api` is a dependency-free shim of the [Logfire](https://logfire.pydantic.dev) API. When the full
`logfire` package is installed, `import logfire_api` returns the `logfire` module and the calls below record
OpenTelemetry spans; without it, the same calls are accepted and do nothing.

Each example checks whichever mode is installed. After `pip install logfire_api` alone, the examples check the
no-op behaviour; after `pip install logfire` as well, they also check the spans captured by an in-memory exporter
(nothing is sent to the Logfire service). Example 2 checks its models only when `pydantic` is installed.

## Example 1: Configuration and structured logging

In [ ]:
import importlib.util
import logfire_api

# Example 1: Configuration and structured logging
print("Example 1: Configuration and Structured Logging")

# With the full logfire package installed, importing logfire_api returns the logfire module itself.
FULL_LOGFIRE = importlib.util.find_spec("logfire") is not None
print(f"logfire installed: {FULL_LOGFIRE}; logfire_api resolves to the {logfire_api.__name__!r} module")
assert logfire_api.__name__ == ("logfire" if FULL_LOGFIRE else "logfire_api"), logfire_api.__name__

exporter = None
if FULL_LOGFIRE:
    # send_to_logfire=False keeps everything local; an in-memory OpenTelemetry exporter captures the spans.
    from opentelemetry.sdk.trace.export import SimpleSpanProcessor
    from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter
    exporter = InMemorySpanExporter()
    instance = logfire_api.configure(send_to_logfire=False, console=False,
                                     additional_span_processors=[SimpleSpanProcessor(exporter)])
else:
    # The shim accepts the same arguments and returns its no-op Logfire instance.
    instance = logfire_api.configure(send_to_logfire=False, console=False)
assert isinstance(instance, logfire_api.Logfire), type(instance)
print(f"✅ configure() returned a {type(instance).__name__} instance")

# Structured logging: keyword arguments become attributes of the log record
print("\n📝 Structured Logging Examples:")
returned = [
    logfire_api.info("User login successful", user_id=12345, session="abc-123"),
    logfire_api.warn("Rate limit approaching", current_requests=95, limit=100, user="john@example.com"),
]
try:
    result = 1 / 0
except ZeroDivisionError as e:
    returned.append(logfire_api.error("Division by zero error", error=str(e), operation="calculate"))
assert returned == [None, None, None], returned  # the logging methods return nothing in either mode
print("  • info(), warn() and error() accepted structured attributes")

if exporter is None:
    print("  Span checks skipped: logfire is not installed, so the shim records nothing (pip install logfire to run them)")
else:
    spans = exporter.get_finished_spans()
    assert [s.name for s in spans] == ["User login successful", "Rate limit approaching", "Division by zero error"], [s.name for s in spans]
    assert spans[0].attributes["user_id"] == 12345, dict(spans[0].attributes)
    # Logfire's default scrubber redacts attributes whose name looks sensitive ('session' is one).
    assert spans[0].attributes["session"].startswith("[Scrubbed"), dict(spans[0].attributes)
    assert spans[2].attributes["operation"] == "calculate", dict(spans[2].attributes)
    assert [s.attributes["logfire.level_num"] for s in spans] == [9, 13, 17], [dict(s.attributes) for s in spans]  # info, warn, error
    print(f"  ✓ {len(spans)} log records captured by the in-memory exporter")

print("Example 1: Configuration and structured logging demonstrated")

## Example 2: Pydantic integration and model validation

In [ ]:
import importlib.util
import logfire_api

# Example 2: Pydantic integration and model validation
print("Example 2: Pydantic Integration and Model Validation")

if importlib.util.find_spec("pydantic") is None:
    # The shim's instrument_pydantic() is a no-op that needs nothing; the full logfire package imports pydantic for it.
    if not FULL_LOGFIRE:
        assert logfire_api.instrument_pydantic() is None
    print("  pydantic is not installed, so the model checks are skipped (pip install pydantic to run them)")
else:
    from pydantic import BaseModel

    # instrument_pydantic() must run before the models are defined: logfire instruments the models created after it.
    assert logfire_api.instrument_pydantic() is None
    print("✅ Pydantic instrumentation enabled")
    if exporter is not None:
        exporter.clear()

    class User(BaseModel):
        id: int
        name: str
        email: str
        age: int

    class OrderRequest(BaseModel):
        user: User
        product_id: str
        quantity: int
        total_amount: float

    user_data = {"id": 1001, "name": "Alice Johnson", "email": "alice@example.com", "age": 28}
    order_data = {"user": user_data, "product_id": "PROD-456", "quantity": 2, "total_amount": 49.99}

    print("\n🔍 Pydantic Validation:")
    user = User(**user_data)
    order = OrderRequest(**order_data)
    print(f"  • User validation: {user.name} ({user.email})")
    print(f"  • Order validation: {order.quantity}x {order.product_id}")
    assert order.user == user, order.user
    assert OrderRequest.model_validate(order.model_dump()) == order  # the validated models round-trip

    if exporter is None:
        print("  Validation-span checks skipped: logfire is not installed (pip install logfire to run them)")
    else:
        validated = {s.attributes.get("schema_name") for s in exporter.get_finished_spans()}
        assert {"User", "OrderRequest"} <= validated, validated
        print(f"  ✓ validation spans recorded for {sorted(name for name in validated if name)}")

print("Example 2: Pydantic integration demonstrated")

## Example 3: Spans and nested tracing

In [ ]:
import time
import logfire_api

# Example 3: Spans and nested tracing
print("Example 3: Spans and Nested Tracing")
if exporter is not None:
    exporter.clear()

def process_order(user_id: int, product_id: str, quantity: int):
    """Process a user order inside a span, with one child span per step."""

    with logfire_api.span("process_user_order", user_id=user_id, product_id=product_id) as span:
        print(f"  🔍 Processing order for user {user_id} (span recording: {span.is_recording()})")
        assert span.is_recording() == FULL_LOGFIRE, span.is_recording()  # the shim's spans never record

        with logfire_api.span("validate_order", user_id=user_id, product_id=product_id):
            time.sleep(0.01)  # Simulate validation time
            print(f"    ✓ Order validation completed")

        with logfire_api.span("process_payment", amount=quantity * 29.99):
            time.sleep(0.02)  # Simulate payment processing
            print(f"    ✓ Payment processed")

        with logfire_api.span("update_inventory", product_id=product_id, quantity=quantity):
            time.sleep(0.01)  # Simulate inventory update
            print(f"    ✓ Inventory updated")

        return {"order_id": "ORD-12345", "status": "completed"}

print("\n🔍 Executing Traced Order Process:")
result = process_order(user_id=1001, product_id="PROD-789", quantity=2)
print(f"  • Order result: {result}")
assert result == {"order_id": "ORD-12345", "status": "completed"}, result

if exporter is None:
    print("  Span checks skipped: logfire is not installed, so the shim records nothing (pip install logfire to run them)")
else:
    spans = {span.name: span for span in exporter.get_finished_spans()}
    assert set(spans) == {"process_user_order", "validate_order", "process_payment", "update_inventory"}, sorted(spans)
    root = spans["process_user_order"]
    assert root.parent is None and root.attributes["user_id"] == 1001, dict(root.attributes)
    assert all(spans[name].parent.span_id == root.context.span_id for name in ("validate_order", "process_payment", "update_inventory"))
    assert spans["process_payment"].attributes["amount"] == 2 * 29.99, dict(spans["process_payment"].attributes)
    print(f"  ✓ {len(spans)} nested spans captured by the in-memory exporter")

print("Example 3: Spans and nested tracing demonstrated")

## Example 4: Function instrumentation with @instrument

In [ ]:
import logfire_api

# Example 4: Function instrumentation with @instrument
print("Example 4: Function Instrumentation with @instrument")
if exporter is not None:
    exporter.clear()

# @instrument wraps each call in a span whose message template is filled from the call's arguments.
@logfire_api.instrument("Computing {base} to the power {exponent}")
def power(base, exponent=2):
    """Raise base to exponent."""
    return base ** exponent

@logfire_api.instrument("Dividing {a} by {b}")
def divide(a, b):
    return a / b

assert power(3) == 9 and power(2, exponent=10) == 1024
assert power.__name__ == "power" and power.__doc__ == "Raise base to exponent.", (power.__name__, power.__doc__)
print("  • power(3) == 9 and power(2, exponent=10) == 1024; the decorated function keeps its name and docstring")
try:
    divide(1, 0)
except ZeroDivisionError:
    print("  • divide(1, 0) raised ZeroDivisionError through the instrumented wrapper")
else:
    raise AssertionError("divide(1, 0) should raise ZeroDivisionError")

if exporter is None:
    print("  Span checks skipped: logfire is not installed, so the shim records nothing (pip install logfire to run them)")
else:
    spans = exporter.get_finished_spans()
    messages = [s.attributes["logfire.msg"] for s in spans]
    assert messages == ["Computing 3 to the power 2", "Computing 2 to the power 10", "Dividing 1 by 0"], messages
    assert [(s.attributes["base"], s.attributes["exponent"]) for s in spans[:2]] == [(3, 2), (2, 10)], [dict(s.attributes) for s in spans[:2]]
    assert spans[2].status.status_code.name == "ERROR", spans[2].status
    assert [event.name for event in spans[2].events] == ["exception"], spans[2].events
    print(f"  ✓ {len(spans)} spans recorded; the failed call has status ERROR and an exception event")

print("Example 4: Function instrumentation demonstrated")